In [1]:
card_names = ['10c', '10d', '10h', '10s', '2c', '2d', '2h', '2s', 
              '3c', '3d', '3h', '3s', '4c', '4d', '4h', '4s',
              '5c', '5d', '5h', '5s', '6c', '6d', '6h', '6s',
              '7c', '7d', '7h', '7s', '8c', '8d', '8h', '8s',
              '9c', '9d', '9h', '9s', 'Ac', 'Ad', 'Ah', 'As',
              'Jc', 'Jd', 'Jh', 'Js', 'Kc', 'Kd', 'Kh', 'Ks',
              'Qc', 'Qd', 'Qh', 'Qs']

In [2]:
def yolo_to_pixels(box_line, img_width, img_height):
    parts = box_line.strip().split()
    class_id = int(parts[0])
    x_center = float(parts[1]) * img_width
    y_center = float(parts[2]) * img_height
    w = float(parts[3]) * img_width
    h = float(parts[4]) * img_height

    x1 = max(0, x_center - w/2)
    y1 = max(0, y_center - h/2)
    x2 = min(img_width, x_center + w/2)
    y2 = min(img_height, y_center + h/2)

    card_name = card_names[class_id]
    return x1, y1, x2, y2, card_name


In [3]:
from PIL import Image
from tqdm import tqdm
import os

input_size = 64
pad = 0.25

source_dirs = {
    'train': './train/images',
    'valid': './valid/images',
    'test': './test/images'
}

label_dirs = {
    'train': './train/labels',
    'valid': './valid/labels',
    'test': './test/labels'
}

base_dir = './cnn_dataset'

# create folders for all splits and cards
for split in ['train', 'valid', 'test']:
    split_dir = os.path.join(base_dir, split)
    os.makedirs(split_dir, exist_ok=True)
    for card in card_names:
        os.makedirs(os.path.join(split_dir, card), exist_ok=True)

for split in ['train', 'valid', 'test']:
    print(f"Processing {split} split...")
    src_dir = source_dirs[split]
    lbl_dir = label_dirs[split]
    out_dir = os.path.join(base_dir, split)

    all_images = [f for f in os.listdir(src_dir) if f.endswith(('.jpg','.png'))]

    for img_name in tqdm(all_images):
        img_path = os.path.join(src_dir, img_name)
        label_path = os.path.join(lbl_dir, img_name.replace('.jpg','.txt'))

        if not os.path.exists(label_path):
            continue

        img = Image.open(img_path)
        W, H = img.size

        with open(label_path,'r') as f:
            lines = f.readlines()

        for i, line in enumerate(lines):
            x1, y1, x2, y2, card_name = yolo_to_pixels(line, W, H)

            # add padding
            w = x2 - x1
            h = y2 - y1
            new_x1 = max(0, x1 - pad*w)
            new_y1 = max(0, y1 - pad*h)
            new_x2 = min(W, x2 + pad*w)
            new_y2 = min(H, y2 + pad*h)

            crop = img.crop((new_x1, new_y1, new_x2, new_y2))
            crop = crop.resize((input_size, input_size))

            save_path = os.path.join(out_dir, card_name, f"{img_name.split('.')[0]}_{i}.jpg")
            crop.save(save_path)

Processing train split...


100%|██████████| 14000/14000 [04:37<00:00, 50.46it/s]


Processing valid split...


100%|██████████| 4000/4000 [01:19<00:00, 50.23it/s]


Processing test split...


100%|██████████| 2000/2000 [00:39<00:00, 50.88it/s]
